# MSSP607 Participation Activity


## Step 1. Import the Required Modules

In [ ]:
import requests
from pathlib import Path
from urllib.parse import urlparse
import re

## Step 2. Choose an Image Category


In [ ]:
CATEGORY = "cats"
NUMBER_OF_IMAGES = 20

print("Image category:", CATEGORY)

## Step 3. Search Wikimedia Commons



In [ ]:
API_URL = "https://commons.wikimedia.org/w/api.php"

params = {
    "action": "query",
    "generator": "search",
    "gsrsearch": CATEGORY,
    "gsrnamespace": 6,
    "gsrlimit": NUMBER_OF_IMAGES,
    "prop": "imageinfo",
    "iiprop": "url",
    "format": "json"
}

headers = {
    "User-Agent": "MSSP607-ImageDownloader/1.0"
}

response = requests.get(
    API_URL,
    params=params,
    headers=headers,
    timeout=30
)

response.raise_for_status()
data = response.json()

pages = data.get("query", {}).get("pages", {})

print("Search completed successfully.")
print("Number of results found:", len(pages))

## Step 4. Create a Folder for the Downloaded Images

In [ ]:
safe_category = re.sub(r"[^A-Za-z0-9_-]+", "_", CATEGORY.strip())

download_folder = Path(f"images_{safe_category}")
download_folder.mkdir(exist_ok=True)

print("Images will be saved in:", download_folder.resolve())

## Step 5. Download the Images



In [ ]:
downloaded_files = []

for index, page in enumerate(pages.values(), start=1):
    imageinfo = page.get("imageinfo", [])

    if not imageinfo:
        continue

    image_url = imageinfo[0]["url"]

    suffix = Path(urlparse(image_url).path).suffix.lower()

    if suffix not in [".jpg", ".jpeg", ".png", ".gif", ".webp"]:
        print(f"Skipped {index}: unsupported file type {suffix}")
        continue

    filename = download_folder / f"{safe_category}_{index:02d}{suffix}"

    try:
        image_response = requests.get(
            image_url,
            headers=headers,
            timeout=30
        )
        image_response.raise_for_status()

        with open(filename, "wb") as image_file:
            for chunk in image_response.iter_content(100000):
                image_file.write(chunk)

        downloaded_files.append(filename)
        print(f"Downloaded {len(downloaded_files)}: {filename.name}")

    except requests.RequestException as error:
        print(f"Could not download image {index}: {error}")

print()
print("Download complete.")
print("Total images downloaded:", len(downloaded_files))

## Step 6. Show the Downloaded Files

In [ ]:
for file in downloaded_files:
    print(file)

## Step 7. Preview the First Five Images

In [ ]:
from IPython.display import display
from PIL import Image

# Preview a few images as thumbnails so the notebook file does not
# become very large when outputs are saved.
for file in downloaded_files[:5]:
    try:
        print(file.name)

        with Image.open(file) as img:
            preview = img.copy()
            preview.thumbnail((500, 500))
            display(preview)

    except Exception as error:
        print("Could not preview:", file.name, error)

## Step 8. Create a ZIP File



In [ ]:
import shutil

zip_name = f"images_{safe_category}"

shutil.make_archive(
    zip_name,
    "zip",
    download_folder
)

print(f"Created {zip_name}.zip")

## Explanation

This program automates a repetitive web task. It first sends a search request to Wikimedia Commons using the category selected by the user. The API returns matching image files and their URLs.

The program then loops through the results and uses `requests.get()` to download each supported image. The files are saved in binary mode inside a new folder. The program also lists the downloaded files, previews several images, and creates a ZIP file.

Instead of manually searching for pictures and saving them one at a time, Python completes the process automatically.